# Clinical Severity-Weighted Hallucination Score (CWHS)

Post-hoc analysis notebook that computes the **Clinical Severity-Weighted Hallucination Score (CWHS)**
across all RAG architectures evaluated in this study.

**Core idea:** Standard hallucination evaluation (e.g. DeepEval FaithfulnessMetric) treats all
questions equally. In medical QA, a hallucinated drug dosage is far more dangerous than a hallucinated
definition. CWHS applies clinical risk weights to the per-question hallucination rate derived from
FaithfulnessMetric, producing a severity-adjusted score that penalises architectures that hallucinate
more on the highest-risk question types.

**Formula:** `CWHS = Σ[(1 - f_i) × w_i] / Σ[w_i]`  
where `f_i` = FaithfulnessMetric score, `w_i` = severity weight (High=3, Medium=2, Low=1)

**Severity classification:** Two-stage hybrid — keyword matching (Stage 1) with LLM fallback
for questions that match no keywords (Stage 2). Labels are cached to
`datasets/processed/golden_dataset_with_severity.csv` and reused across all architectures.

In [4]:
import sys
sys.path.append("..")

import os
import time
import glob
from ast import literal_eval
import pandas as pd
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
import config

import logging
logging.basicConfig(level=logging.ERROR)

## Severity Classification Constants

Clinical risk taxonomy for biomedical questions:
- **High (weight=3)**: Treatment, medication, dosage, surgery — wrong answer could directly harm a patient
- **Medium (weight=2)**: Diagnosis, symptoms, prognosis, risk factors — wrong answer could mislead clinical reasoning
- **Low (weight=1)**: Definitions, mechanisms, epidemiology — primarily educational, indirect patient impact

In [6]:
HIGH_RISK_KEYWORDS = [
    "drug", "dose", "dosage", "medication", "prescri", "antibiotic",
    "treat", "treatment", "therapy", "therapies", "intervention", "regimen",
    "surgery", "surgical", "procedure", "adverse", "vaccine", "vaccination",
]

MEDIUM_RISK_KEYWORDS = [
    "diagnos", "symptom", "prognosis", "outcome", "risk factor", "risk of",
    "predict", "indicator", "biomarker", "complication",
    "disease", "disorder", "syndrome", "condition", "side effect",
]

SEVERITY_WEIGHTS = {"High": 3, "Medium": 2, "Low": 1}

## Stage 1: Keyword-Based Classification

In [7]:
def classify_by_keyword(question: str):
    """Stage 1: classify question severity using keyword substring matching.

    Returns:
        Tuple of (tier str, source str) where source is 'keyword' if matched,
        or 'pending_llm' if no keywords matched (requires Stage 2).
    """
    q = question.lower()
    if any(kw in q for kw in HIGH_RISK_KEYWORDS):
        return "High", "keyword"
    if any(kw in q for kw in MEDIUM_RISK_KEYWORDS):
        return "Medium", "keyword"
    return "Low", "pending_llm"

## Stage 2: LLM Fallback for Keyword-Unmatched Questions

Any question that matched no keywords in Stage 1 is classified by a single LLM call.
This handles biomedical phrasing that does not surface the expected keywords
(Latin terminology, abbreviations, paraphrases). The LLM is given the same
three-tier taxonomy with concrete clinical examples for each tier.

Stage 2 calls are parallelised across API keys using the same `ThreadPoolExecutor`
pattern used throughout the project.

In [8]:
SEVERITY_LLM_PROMPT_TEMPLATE = """You are a clinical risk assessor for medical AI systems.
Classify the following biomedical question into exactly one clinical risk tier.

Risk tiers:
- High: The question concerns drug dosage, medication choice, treatment protocol, surgical
  procedure, or any clinical decision that directly affects patient care. A wrong answer
  could cause direct patient harm.
  Examples: "Which antibiotic should be used?", "What is the standard dose of metformin?",
  "Is surgery indicated for this condition?"

- Medium: The question concerns diagnosis, symptom interpretation, prognosis, risk factors,
  or clinical outcome prediction. A wrong answer could mislead clinical reasoning but
  does not directly prescribe an action.
  Examples: "What are the early symptoms of Parkinson's?", "Does smoking increase the risk
  of this disease?", "What is the prognosis for stage 2 lung cancer?"

- Low: The question concerns definitions, biological mechanisms, epidemiology, or general
  scientific knowledge. A wrong answer is primarily misleading in an educational context.
  Examples: "What is the mechanism of action of aspirin?", "How common is type 1 diabetes?",
  "What is the role of the hippocampus in memory?"

Return only one word: High, Medium, or Low. Do not explain.

Question: {question}

Risk tier:"""

SEVERITY_LLM_PROMPT = PromptTemplate(
    template=SEVERITY_LLM_PROMPT_TEMPLATE,
    input_variables=["question"],
)


class GroqKeyRotator:
    def __init__(self, model=None):
        if not config.GROQ_API_KEYS:
            raise ValueError("config.GROQ_API_KEYS is empty")
        self.api_keys = config.GROQ_API_KEYS
        self.model = model or config.LLM_MODEL
        print(f"Initialized GroqKeyRotator with {len(self.api_keys)} API key(s)")

    def get_llm(self):
        return ChatGroq(model=self.model, api_key=self.api_keys[0])


def _classify_slice_llm(questions_with_idx, api_key, model, delay, key_idx):
    """Classify a slice of questions via LLM. Returns list of (idx, tier) tuples."""
    llm = ChatGroq(model=model, api_key=api_key)
    chain = SEVERITY_LLM_PROMPT | llm
    results = []
    for i, (orig_idx, question) in enumerate(questions_with_idx):
        try:
            raw = chain.invoke({"question": question}).content.strip()
            tier = raw if raw in ("High", "Medium", "Low") else "Low"
        except Exception as e:
            print(f"[Key {key_idx}] Error on question {orig_idx}: {e}")
            tier = "Low"
        results.append((orig_idx, tier))
        if i < len(questions_with_idx) - 1:
            time.sleep(delay)
    print(f"[Key {key_idx}] Done — {len(results)} questions classified")
    return results


def classify_llm_parallel(questions_with_idx, key_rotator, rows_per_key=None, delay=None):
    """Classify questions via LLM in parallel across API keys.

    Args:
        questions_with_idx: List of (original_index, question_text) tuples.
        key_rotator: GroqKeyRotator instance.
        rows_per_key: Max questions per key (default config.PARALLEL_BUCKET_SIZE).
        delay: Seconds between questions per key.

    Returns:
        Dict mapping original_index → tier string.
    """
    rows_per_key = rows_per_key or config.PARALLEL_BUCKET_SIZE
    delay = delay or 1  # LLM classification is a short call; 1s delay is sufficient
    api_keys = key_rotator.api_keys

    slices = []
    for i, key in enumerate(api_keys):
        start = i * rows_per_key
        if start >= len(questions_with_idx):
            break
        slices.append((key, i, questions_with_idx[start: start + rows_per_key]))

    print(f"\n{len(questions_with_idx)} questions for LLM fallback, split across {len(slices)} key(s):")

    ordered_results = [None] * len(slices)
    with ThreadPoolExecutor(max_workers=len(slices)) as executor:
        future_to_idx = {
            executor.submit(_classify_slice_llm, s, key, key_rotator.model, delay, i): idx
            for idx, (key, i, s) in enumerate(slices)
        }
        for future in as_completed(future_to_idx):
            slice_idx = future_to_idx[future]
            try:
                ordered_results[slice_idx] = future.result()
            except Exception as e:
                print(f"[Key {slice_idx}] Thread failed: {e}")
                ordered_results[slice_idx] = []

    result_map = {}
    for key_results in ordered_results:
        for orig_idx, tier in key_results:
            result_map[orig_idx] = tier
    return result_map

## Hybrid Classifier + Caching

Runs Stage 1 (keyword) for all questions, then Stage 2 (LLM) for unmatched ones.
Results are saved to `datasets/processed/golden_dataset_with_severity.csv` and
reloaded on subsequent runs — the LLM calls only fire once for the 200-question set.

In [9]:
SEVERITY_CACHE_PATH = config.DATA_PROCESSED_DIR / "golden_dataset_with_severity.csv"


def build_severity_labels(golden_df, key_rotator, force_recompute=False):
    """Classify all questions by clinical severity using the hybrid approach.

    Loads from cache if it exists (unless force_recompute=True).
    Stage 1: keyword matching for all questions.
    Stage 2: LLM fallback for questions that matched no keywords.

    Args:
        golden_df: DataFrame with at least a 'question' column.
        key_rotator: GroqKeyRotator for Stage 2 LLM calls.
        force_recompute: If True, ignore cache and re-run classification.

    Returns:
        DataFrame with added columns: severity_tier, severity_weight, classification_source.
    """
    if not force_recompute and SEVERITY_CACHE_PATH.exists():
        print(f"Loading severity labels from cache: {SEVERITY_CACHE_PATH}")
        cached = pd.read_csv(SEVERITY_CACHE_PATH)
        return cached

    df = golden_df.copy().reset_index(drop=True)

    # Stage 1: keyword classification
    tiers, sources = [], []
    for q in df["question"]:
        tier, source = classify_by_keyword(q)
        tiers.append(tier)
        sources.append(source)

    df["severity_tier"] = tiers
    df["classification_source"] = sources

    pending_llm = df[df["classification_source"] == "pending_llm"]
    keyword_count = len(df) - len(pending_llm)
    print(f"Stage 1 (keyword): {keyword_count}/{len(df)} questions classified")
    print(f"Stage 2 (LLM fallback): {len(pending_llm)} questions to reclassify")

    # Stage 2: LLM reclassification for pending_llm questions
    if len(pending_llm) > 0:
        questions_with_idx = list(zip(pending_llm.index.tolist(), pending_llm["question"].tolist()))
        llm_results = classify_llm_parallel(questions_with_idx, key_rotator)
        for orig_idx, tier in llm_results.items():
            df.at[orig_idx, "severity_tier"] = tier
            df.at[orig_idx, "classification_source"] = "llm_fallback"

    df["severity_weight"] = df["severity_tier"].map(SEVERITY_WEIGHTS)

    # Cache results
    df.to_csv(SEVERITY_CACHE_PATH, index=False)
    print(f"\nSeverity labels saved to {SEVERITY_CACHE_PATH}")

    # Summary
    print(f"\nSeverity distribution:")
    for tier in ["High", "Medium", "Low"]:
        n = (df["severity_tier"] == tier).sum()
        print(f"  {tier} (w={SEVERITY_WEIGHTS[tier]}): {n} questions ({100*n/len(df):.1f}%)")
    print(f"\nClassification source:")
    print(df["classification_source"].value_counts().to_string())

    return df

## CWHS and CSS Computation

Two complementary metrics:
- **CWHS** (Clinical Severity-Weighted Hallucination Score): uses faithfulness only — measures
  whether hallucinations concentrate on high-risk questions.
- **CSS** (Clinical Safety Score): combines faithfulness and answer correctness with severity
  weights — captures both hallucination risk and answer quality in a single score.

In [10]:
CSS_ALPHA = 0.6  # Weight for hallucination (faithfulness) component
CSS_BETA = 0.4   # Weight for incorrectness (answer correctness) component


def compute_cwhs(severity_df, faithfulness_scores):
    """Compute Clinical Severity-Weighted Hallucination Score.

    CWHS = Σ[(1 - f_i) × w_i] / Σ[w_i]

    Args:
        severity_df: DataFrame with severity_tier and severity_weight columns,
            aligned by index to faithfulness_scores.
        faithfulness_scores: List or Series of FaithfulnessMetric scores (0–1,
            higher = less hallucination).

    Returns:
        Dict with:
          cwhs: float [0-1], severity-weighted hallucination score
          unweighted_hal_rate: float [0-1], plain 1 - mean(faithfulness)
          delta: cwhs - unweighted_hal_rate (positive = worse on high-risk Qs)
          mean_faithfulness: float, mean faithfulness score
          n_questions: int, number of questions evaluated
          severity_breakdown: per-tier dict with count and mean hallucination rate
    """
    weights = list(severity_df["severity_weight"])
    tiers = list(severity_df["severity_tier"])
    scores = list(faithfulness_scores)

    assert len(weights) == len(scores), "Mismatch between severity labels and faithfulness scores"

    hal_rates = [1 - f for f in scores]
    total_weight = sum(weights)
    cwhs = sum(h * w for h, w in zip(hal_rates, weights)) / total_weight
    unweighted = sum(hal_rates) / len(hal_rates)

    breakdown = {}
    for tier in ["High", "Medium", "Low"]:
        indices = [i for i, t in enumerate(tiers) if t == tier]
        if indices:
            tier_hal = [hal_rates[i] for i in indices]
            breakdown[tier] = {
                "count": len(indices),
                "mean_hal_rate": round(sum(tier_hal) / len(tier_hal), 4),
                "weight": SEVERITY_WEIGHTS[tier],
            }
        else:
            breakdown[tier] = {"count": 0, "mean_hal_rate": None, "weight": SEVERITY_WEIGHTS[tier]}

    return {
        "cwhs": round(cwhs, 4),
        "unweighted_hal_rate": round(unweighted, 4),
        "delta": round(cwhs - unweighted, 4),
        "mean_faithfulness": round(sum(scores) / len(scores), 4),
        "n_questions": len(scores),
        "severity_breakdown": breakdown,
    }


def compute_css(severity_df, faithfulness_scores, correctness_scores, alpha=None, beta=None):
    """Compute Clinical Safety Score combining faithfulness and answer correctness.

    CSS = [α × Σ((1 - f_i) × w_i) + β × Σ((1 - c_i) × w_i)] / Σ[w_i]

    where f_i = faithfulness score, c_i = answer correctness score,
    w_i = severity weight (High=3, Medium=2, Low=1),
    α = hallucination penalty weight, β = incorrectness penalty weight.

    Lower CSS = safer architecture. CSS captures both dimensions:
    an architecture that is faithful to wrong chunks (high faithfulness, low correctness)
    is penalised by the β term; one that goes beyond its context but gets correct answers
    is penalised by the α term.

    Args:
        severity_df: DataFrame with severity_tier and severity_weight columns.
        faithfulness_scores: List of FaithfulnessMetric scores (0–1).
        correctness_scores: List of AnswerCorrectness scores (0–1).
        alpha: Hallucination penalty weight (default CSS_ALPHA).
        beta: Incorrectness penalty weight (default CSS_BETA).

    Returns:
        Dict with:
          css: float, combined clinical safety score (lower is better)
          mean_correctness: float
          correctness_breakdown: per-tier dict with count and mean error rate
          faith_p10: float, 10th percentile faithfulness (tail risk)
          corr_p10: float, 10th percentile correctness (tail risk)
          n_questions: int
    """
    alpha = alpha if alpha is not None else CSS_ALPHA
    beta = beta if beta is not None else CSS_BETA

    weights = list(severity_df["severity_weight"])
    tiers = list(severity_df["severity_tier"])
    f_scores = list(faithfulness_scores)
    c_scores = list(correctness_scores)

    assert len(weights) == len(f_scores) == len(c_scores), \
        f"Length mismatch: weights={len(weights)}, faith={len(f_scores)}, corr={len(c_scores)}"

    total_weight = sum(weights)
    hal_component = sum((1 - f) * w for f, w in zip(f_scores, weights))
    corr_component = sum((1 - c) * w for c, w in zip(c_scores, weights))
    css = (alpha * hal_component + beta * corr_component) / total_weight

    import numpy as np
    f_arr = np.array(f_scores)
    c_arr = np.array(c_scores)

    corr_breakdown = {}
    for tier in ["High", "Medium", "Low"]:
        indices = [i for i, t in enumerate(tiers) if t == tier]
        if indices:
            tier_err = [1 - c_scores[i] for i in indices]
            tier_corr = [c_scores[i] for i in indices]
            corr_breakdown[tier] = {
                "count": len(indices),
                "mean_error_rate": round(sum(tier_err) / len(tier_err), 4),
                "mean_correctness": round(sum(tier_corr) / len(tier_corr), 4),
                "weight": SEVERITY_WEIGHTS[tier],
            }
        else:
            corr_breakdown[tier] = {
                "count": 0, "mean_error_rate": None,
                "mean_correctness": None, "weight": SEVERITY_WEIGHTS[tier],
            }

    return {
        "css": round(css, 4),
        "mean_correctness": round(sum(c_scores) / len(c_scores), 4),
        "correctness_breakdown": corr_breakdown,
        "faith_p10": round(float(np.percentile(f_arr, 10)), 4),
        "corr_p10": round(float(np.percentile(c_arr, 10)), 4),
        "n_questions": len(f_scores),
    }

## Architecture Auto-Discovery

Scans `results/deepeval/` for faithfulness and answer correctness CSVs. Each architecture
has exactly one faithfulness CSV (matching `*_faithfulness_*.csv`) and at most one answer
correctness CSV (matching `*_answer_correctness_*.csv`). The prefix before `_faithfulness_`
or `_answer_correctness_` uniquely identifies the architecture and embedding key combination.

Architectures without a faithfulness file (e.g. Vanilla LLM with no retrieval context) are
excluded from CWHS/CSS computation since both metrics require faithfulness scores.

In [19]:
DEEPEVAL_DIR = config.RESULTS_DEEPEVAL_DIR

PREFIX_TO_LABEL = {
    "naive_rag_minilm":                   "Naive RAG k=3",
    "naive_rag_minilm_k_5":              "Naive RAG k=5",
    "naive_rag_minilm_k_8":              "Naive RAG k=8",
    "naive_rag_critic_minilm":                  "Naive RAG With Critic",
    "query_expansion_rag_minilm":         "Query Expansion (Single)",
    "multi_query_expansion_rag_minilm":   "Query Expansion (Multi)",
    "hybrid_rrf_rag_minilm":              "Hybrid RRF",
    "hybrid_cross_encoder_rag_minilm":    "Hybrid + Cross-Encoder",
    "decomposition_rag_minilm":           "Query Decomposition",
    "mesh_guided_rag_minilm":             "MeSH-Guided RAG",
    "evidence_graded_rag_minilm":         "Evidence-Graded RAG",
}


def discover_architectures(deepeval_dir):
    """Auto-discover architecture CSVs from the deepeval results directory.

    Scans for *_faithfulness_*.csv and *_answer_correctness_*.csv files.
    Groups them by prefix (everything before _faithfulness_ or _answer_correctness_).
    Each prefix maps to at most one file per metric type.

    Returns:
        Dict mapping human-readable label to dict with keys:
          'faithfulness_file': filename or None
          'correctness_file': filename or None
          'prefix': raw file prefix
    """
    all_files = os.listdir(deepeval_dir)

    faith_files = {}
    corr_files = {}
    for f in all_files:
        if "_faithfulness_" in f and f.endswith(".csv"):
            prefix = f.split("_faithfulness_")[0]
            faith_files[prefix] = f
        elif "_answer_correctness_" in f and f.endswith(".csv"):
            prefix = f.split("_answer_correctness_")[0]
            corr_files[prefix] = f

    architectures = {}
    for prefix in sorted(faith_files.keys()):
        label = PREFIX_TO_LABEL.get(prefix, prefix)
        architectures[label] = {
            "faithfulness_file": faith_files[prefix],
            "correctness_file": corr_files.get(prefix),
            "prefix": prefix,
        }

    return architectures


def load_metric_csv(filename, metric_keyword):
    """Load a DeepEval metric CSV and return (questions, scores, question_idxs).

    Args:
        filename: CSV filename (relative to DEEPEVAL_DIR).
        metric_keyword: Substring to match in column name (e.g. 'faithfulness', 'correct').

    Returns:
        Tuple of (questions list, scores list, question_idxs list or None),
        or None if file doesn't exist.
    """
    path = DEEPEVAL_DIR / filename
    if not path.exists():
        return None
    df = pd.read_csv(path)
    score_col = next((c for c in df.columns if metric_keyword in c.lower()), None)
    if score_col is None:
        print(f"Warning: no '{metric_keyword}' column found in {filename}")
        return None
    q_idxs = df["question_idx"].tolist() if "question_idx" in df.columns else None
    return df["question"].tolist(), df[score_col].tolist(), q_idxs


ARCHITECTURES = discover_architectures(DEEPEVAL_DIR)

print(f"Discovered {len(ARCHITECTURES)} architectures in {DEEPEVAL_DIR}:\n")
for label, info in ARCHITECTURES.items():
    faith = "+" if info["faithfulness_file"] else "-"
    corr = "+" if info["correctness_file"] else "-"
    print(f"  {label:35s}  faithfulness={faith}  correctness={corr}")

Discovered 11 architectures in /content/results/deepeval:

  Query Decomposition                  faithfulness=+  correctness=+
  Evidence-Graded RAG                  faithfulness=+  correctness=+
  Hybrid + Cross-Encoder               faithfulness=+  correctness=+
  Hybrid RRF                           faithfulness=+  correctness=+
  MeSH-Guided RAG                      faithfulness=+  correctness=+
  Query Expansion (Multi)              faithfulness=+  correctness=+
  Naive RAG With Critic                faithfulness=+  correctness=+
  Naive RAG k=3                        faithfulness=+  correctness=+
  Naive RAG k=5                        faithfulness=+  correctness=+
  Naive RAG k=8                        faithfulness=+  correctness=+
  Query Expansion (Single)             faithfulness=+  correctness=+


In [20]:
for label, info in ARCHITECTURES.items():
    loaded = load_metric_csv(info["faithfulness_file"], "faithfulness")
    if loaded is None:
        continue
    questions, scores, q_idxs = loaded
    idx_str = f" (question_idx: {'yes' if q_idxs else 'no'})"
    corr_str = ""
    if info["correctness_file"]:
        corr_loaded = load_metric_csv(info["correctness_file"], "correct")
        if corr_loaded:
            _, c_scores, c_idxs = corr_loaded
            corr_str = f", correctness={len(c_scores)} scores (question_idx: {'yes' if c_idxs else 'no'})"
    print(f"{label}: {len(questions)} questions, {len(scores)} faithfulness scores{idx_str}{corr_str}")

Query Decomposition: 200 questions, 200 faithfulness scores (question_idx: yes), correctness=200 scores (question_idx: yes)
Evidence-Graded RAG: 200 questions, 200 faithfulness scores (question_idx: yes), correctness=200 scores (question_idx: yes)
Hybrid + Cross-Encoder: 200 questions, 200 faithfulness scores (question_idx: yes), correctness=200 scores (question_idx: no)
Hybrid RRF: 200 questions, 200 faithfulness scores (question_idx: yes), correctness=200 scores (question_idx: yes)
MeSH-Guided RAG: 200 questions, 200 faithfulness scores (question_idx: yes), correctness=200 scores (question_idx: no)
Query Expansion (Multi): 200 questions, 200 faithfulness scores (question_idx: yes), correctness=200 scores (question_idx: yes)
Naive RAG With Critic: 200 questions, 200 faithfulness scores (question_idx: yes), correctness=200 scores (question_idx: yes)
Naive RAG k=3: 200 questions, 200 faithfulness scores (question_idx: no), correctness=200 scores (question_idx: yes)
Naive RAG k=5: 200 qu

## Setup: Load Golden Dataset and Run Severity Classification

In [14]:
golden_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_complete.csv")
golden_df['golden_contexts'] = golden_df['golden_contexts'].apply(literal_eval)
print(f"Loaded {len(golden_df)} questions from golden_dataset_complete.csv")
golden_df.head(3)

Loaded 200 questions from golden_dataset_complete.csv


,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed
0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841']
1,1,"Do the changes in the serum levels of IL-2, IL...",The enhancement of serum TNFalpha and IL-6 lev...,[Acute pancreatitis is the major complication ...,Single-hop,['18670651']
2,2,Does hypoglycaemia increase the risk of cardio...,Severe hypoglycaemia is associated with an inc...,[Hypoglycaemia caused by glucose-lowering ther...,Single-hop,['23999452']


In [ ]:
# Run severity classification (cached after first run)
key_rotator = GroqKeyRotator()
severity_df = build_severity_labels(golden_df, key_rotator)

Initialized GroqKeyRotator with 10 API key(s)
Stage 1 (keyword): 104/200 questions classified
Stage 2 (LLM fallback): 96 questions to reclassify

96 questions for LLM fallback, split across 5 key(s):
[Key 4] Done — 16 questions classified
[Key 3] Done — 20 questions classified
[Key 1] Done — 20 questions classified
[Key 0] Done — 20 questions classified
[Key 2] Done — 20 questions classified

Severity labels saved to /content/drive/MyDrive/LJMU/processed/golden_dataset_with_severity.csv

Severity distribution:
  High (w=3): 56 questions (28.0%)
  Medium (w=2): 122 questions (61.0%)
  Low (w=1): 22 questions (11.0%)

Classification source:
classification_source
keyword         104
llm_fallback     96


In [ ]:
# Inspect the severity distribution
print("Severity distribution:")
print(severity_df["severity_tier"].value_counts().to_string())
print("\nClassification source:")
print(severity_df["classification_source"].value_counts().to_string())
severity_df[["question", "severity_tier", "severity_weight", "classification_source"]].head(10)

Severity distribution:
severity_tier
Medium    122
High       56
Low        22

Classification source:
classification_source
keyword         104
llm_fallback     96


,question,severity_tier,severity_weight,classification_source
0,Is there a relationship between rheumatoid art...,Medium,2,keyword
1,"Do the changes in the serum levels of IL-2, IL...",Medium,2,llm_fallback
2,Does hypoglycaemia increase the risk of cardio...,Medium,2,keyword
3,Telemedicine and type 1 diabetes: is technolog...,Medium,2,llm_fallback
4,Can elevated troponin I levels predict complic...,Medium,2,keyword
5,Does β-catenin have a role in pathogenesis of ...,Low,1,llm_fallback
6,Remote ischemic postconditioning: does it prot...,Medium,2,keyword
7,Hearing loss: an unknown complication of pre-e...,Medium,2,keyword
8,Does psychological distress predict disability?,Medium,2,keyword
9,Pancreas retransplantation: a second chance f...,High,3,llm_fallback


## Compute CWHS and CSS for All Architectures

For each discovered architecture:
1. Load its faithfulness CSV (required for CWHS)
2. Load its answer correctness CSV if available (required for CSS)
3. Match questions to severity labels by question text
4. Compute CWHS (faithfulness-only) and CSS (combined) using aligned severity weights

In [21]:
severity_df = pd.read_csv(config.DATA_PROCESSED_DIR / "golden_dataset_with_severity.csv")
severity_df['golden_contexts'] = severity_df['golden_contexts'].apply(literal_eval)
severity_df.head(3)

,question_idx,question,golden_answer,golden_contexts,query_type,pubids_needed,severity_tier,classification_source,severity_weight
0,0,Is there a relationship between rheumatoid art...,Based on data derived from self-reported healt...,"[1,412 individuals attending the University of...",Single-hop,['10783841'],Medium,keyword,2
1,1,"Do the changes in the serum levels of IL-2, IL...",The enhancement of serum TNFalpha and IL-6 lev...,[Acute pancreatitis is the major complication ...,Single-hop,['18670651'],Medium,llm_fallback,2
2,2,Does hypoglycaemia increase the risk of cardio...,Severe hypoglycaemia is associated with an inc...,[Hypoglycaemia caused by glucose-lowering ther...,Single-hop,['23999452'],Medium,keyword,2


In [22]:
def align_severity_to_scores(severity_df, questions, scores):
    """Align severity labels to a metric CSV's row order.

    Matches rows by question text (case-insensitive strip). Returns a
    filtered severity DataFrame aligned to the score list.
    Rows whose question is not found in severity_df are assigned default Low severity.
    """
    sev_map = {
        q.strip().lower(): (tier, w)
        for q, tier, w in zip(
            severity_df["question"],
            severity_df["severity_tier"],
            severity_df["severity_weight"],
        )
    }
    aligned_tiers, aligned_weights, aligned_scores = [], [], []
    missing = 0
    for q, s in zip(questions, scores):
        key = q.strip().lower()
        if key in sev_map:
            tier, w = sev_map[key]
        else:
            tier, w = "Low", 1
            missing += 1
        aligned_tiers.append(tier)
        aligned_weights.append(w)
        aligned_scores.append(s)
    if missing:
        print(f"Warning: {missing} questions not found in severity_df — assigned Low by default")
    aligned_df = pd.DataFrame({"severity_tier": aligned_tiers, "severity_weight": aligned_weights})
    return aligned_df, aligned_scores


def _resolve_question_idx(f_questions, f_q_idxs, severity_df):
    """Resolve question_idx for each question in faithfulness order.

    Priority: faithfulness CSV's question_idx > severity_df's question_idx > positional index.
    """
    sev_idx_map = {}
    if "question_idx" in severity_df.columns:
        sev_idx_map = {
            q.strip().lower(): int(idx)
            for q, idx in zip(severity_df["question"], severity_df["question_idx"])
        }

    resolved = []
    for i, q in enumerate(f_questions):
        if f_q_idxs is not None:
            resolved.append(int(f_q_idxs[i]))
        elif q.strip().lower() in sev_idx_map:
            resolved.append(sev_idx_map[q.strip().lower()])
        else:
            resolved.append(i)
    return resolved


cwhs_results = {}
css_results = {}
per_question_dfs = {}
skipped = []

for arch_label, arch_info in ARCHITECTURES.items():
    # Load faithfulness (required)
    faith_loaded = load_metric_csv(arch_info["faithfulness_file"], "faithfulness")
    if faith_loaded is None:
        skipped.append(arch_label)
        continue
    f_questions, f_scores, f_q_idxs = faith_loaded
    aligned_sev_df, aligned_f_scores = align_severity_to_scores(severity_df, f_questions, f_scores)

    # Compute CWHS
    cwhs_result = compute_cwhs(aligned_sev_df, aligned_f_scores)
    cwhs_results[arch_label] = cwhs_result

    # Resolve question_idx
    q_idxs = _resolve_question_idx(f_questions, f_q_idxs, severity_df)

    # Build per-question DataFrame
    pq_df = pd.DataFrame({
        "question_idx": q_idxs,
        "question": f_questions,
        "severity_tier": aligned_sev_df["severity_tier"].tolist(),
        "severity_weight": aligned_sev_df["severity_weight"].tolist(),
        "faithfulness": aligned_f_scores,
        "hallucination_rate": [round(1 - f, 4) for f in aligned_f_scores],
        "cwhs_contribution": [round((1 - f) * w, 4) for f, w in zip(aligned_f_scores, aligned_sev_df["severity_weight"])],
    })

    # Load answer correctness (optional — needed for CSS)
    css_result = None
    if arch_info["correctness_file"]:
        corr_loaded = load_metric_csv(arch_info["correctness_file"], "correct")
        if corr_loaded:
            c_questions, c_scores, c_q_idxs = corr_loaded

            # Build correctness lookup — prefer question_idx if both CSVs have it
            if c_q_idxs is not None and f_q_idxs is not None:
                corr_by_idx = {int(idx): s for idx, s in zip(c_q_idxs, c_scores)}
                aligned_c_scores = [corr_by_idx.get(int(qi), 0.0) for qi in q_idxs]
            else:
                corr_map = {q.strip().lower(): s for q, s in zip(c_questions, c_scores)}
                aligned_c_scores = [corr_map.get(q.strip().lower(), 0.0) for q in f_questions]

            css_result = compute_css(aligned_sev_df, aligned_f_scores, aligned_c_scores)
            css_results[arch_label] = css_result

            pq_df["correctness"] = aligned_c_scores
            pq_df["css_contribution"] = [
                round(CSS_ALPHA * (1 - f) * w + CSS_BETA * (1 - c) * w, 4)
                for f, c, w in zip(aligned_f_scores, aligned_c_scores, aligned_sev_df["severity_weight"])
            ]

    pq_df = pq_df.sort_values("question_idx").reset_index(drop=True)
    per_question_dfs[arch_label] = pq_df

    css_str = f", CSS={css_result['css']:.4f}" if css_result else ""
    print(f"{arch_label}: CWHS={cwhs_result['cwhs']:.4f}, "
          f"Unweighted={cwhs_result['unweighted_hal_rate']:.4f}, "
          f"Δ={cwhs_result['delta']:+.4f}{css_str}")

if skipped:
    print(f"\nSkipped (faithfulness file not found): {', '.join(skipped)}")
print(f"\nCWHS computed for {len(cwhs_results)} architectures")
print(f"CSS computed for {len(css_results)} architectures (requires both faithfulness + correctness)")
print(f"Per-question DataFrames built for {len(per_question_dfs)} architectures")

Query Decomposition: CWHS=0.0163, Unweighted=0.0151, Δ=+0.0012, CSS=0.1104
Evidence-Graded RAG: CWHS=0.0162, Unweighted=0.0168, Δ=-0.0006, CSS=0.0828
Hybrid + Cross-Encoder: CWHS=0.0195, Unweighted=0.0180, Δ=+0.0014, CSS=0.1211
Hybrid RRF: CWHS=0.0198, Unweighted=0.0186, Δ=+0.0013, CSS=0.1392
MeSH-Guided RAG: CWHS=0.0211, Unweighted=0.0189, Δ=+0.0021, CSS=0.1492
Query Expansion (Multi): CWHS=0.0132, Unweighted=0.0119, Δ=+0.0013, CSS=0.1168
Naive RAG With Critic: CWHS=0.0552, Unweighted=0.0547, Δ=+0.0005, CSS=0.1701
Naive RAG k=3: CWHS=0.0148, Unweighted=0.0141, Δ=+0.0007, CSS=0.1443
Naive RAG k=5: CWHS=0.0200, Unweighted=0.0204, Δ=-0.0003, CSS=0.1283
Naive RAG k=8: CWHS=0.0137, Unweighted=0.0148, Δ=-0.0011, CSS=0.1434
Query Expansion (Single): CWHS=0.0141, Unweighted=0.0145, Δ=-0.0003, CSS=0.1428

CWHS computed for 11 architectures
CSS computed for 11 architectures (requires both faithfulness + correctness)
Per-question DataFrames built for 11 architectures


## Summary Table: CWHS, CSS, and Tail Risk Metrics

In [23]:
if cwhs_results:
    summary_rows = []
    for arch in cwhs_results:
        r = cwhs_results[arch]
        row = {
            "Architecture": arch,
            "Mean Faithfulness": r["mean_faithfulness"],
            "Unweighted Hal. Rate": r["unweighted_hal_rate"],
            "CWHS": r["cwhs"],
            "Δ (CWHS − Unweighted)": r["delta"],
            "N Questions": r["n_questions"],
        }
        if arch in css_results:
            cr = css_results[arch]
            row["Mean Correctness"] = cr["mean_correctness"]
            row["CSS"] = cr["css"]
            row["Faith P10"] = cr["faith_p10"]
            row["Corr P10"] = cr["corr_p10"]
        else:
            row["Mean Correctness"] = None
            row["CSS"] = None
            row["Faith P10"] = None
            row["Corr P10"] = None
        summary_rows.append(row)

    summary_df = pd.DataFrame(summary_rows)
    # Sort by CSS where available, then by CWHS
    summary_df = summary_df.sort_values(
        by=["CSS", "CWHS"], ascending=True, na_position="last"
    )
    print("\nCombined Summary (sorted by CSS, lower is better):")
    print(summary_df.to_string(index=False))
    summary_df.to_csv(
        config.RESULTS_FIGURES_DIR / "cwhs_summary.csv", index=False
    )
    print(f"\nSaved summary to {config.RESULTS_FIGURES_DIR / 'cwhs_summary.csv'}")


Combined Summary (sorted by CSS, lower is better):
            Architecture  Mean Faithfulness  Unweighted Hal. Rate   CWHS  Δ (CWHS − Unweighted)  N Questions  Mean Correctness    CSS  Faith P10  Corr P10
     Evidence-Graded RAG             0.9832                0.0168 0.0162                -0.0006          200            0.8275 0.0828     0.9159      0.30
     Query Decomposition             0.9849                0.0151 0.0163                 0.0012          200            0.7475 0.1104     1.0000      0.20
 Query Expansion (Multi)             0.9881                0.0119 0.0132                 0.0013          200            0.7385 0.1168     1.0000      0.20
  Hybrid + Cross-Encoder             0.9820                0.0180 0.0195                 0.0014          200            0.7355 0.1211     1.0000      0.20
           Naive RAG k=5             0.9796                0.0204 0.0200                -0.0003          200            0.7150 0.1283     1.0000      0.20
              Hybr

## Severity-Stratified Breakdown

Shows the hallucination rate and correctness within each severity tier per architecture.
The key finding is whether architectures differ not just in overall rates but in *where*
they fail — an architecture that hallucinates heavily or gives incorrect answers on
High-risk questions is more clinically dangerous than one that makes errors on Low-risk questions.

In [24]:
if cwhs_results:
    breakdown_rows = []
    for arch in cwhs_results:
        r = cwhs_results[arch]
        row = {"Architecture": arch}
        for tier in ["High", "Medium", "Low"]:
            bd = r["severity_breakdown"].get(tier, {})
            row[f"{tier} Hal. Rate"] = bd.get("mean_hal_rate", None)
            row[f"{tier} N"] = bd.get("count", 0)
        row["CWHS"] = r["cwhs"]
        if arch in css_results:
            cr = css_results[arch]
            for tier in ["High", "Medium", "Low"]:
                cbd = cr["correctness_breakdown"].get(tier, {})
                row[f"{tier} Corr."] = cbd.get("mean_correctness", None)
            row["CSS"] = cr["css"]
        else:
            for tier in ["High", "Medium", "Low"]:
                row[f"{tier} Corr."] = None
            row["CSS"] = None
        breakdown_rows.append(row)

    breakdown_df = pd.DataFrame(breakdown_rows).sort_values(
        by=["CSS", "CWHS"], ascending=True, na_position="last"
    )
    print("\nSeverity-Stratified Breakdown:")
    print(breakdown_df.to_string(index=False))
    breakdown_df.to_csv(
        config.RESULTS_FIGURES_DIR / "cwhs_breakdown_by_severity.csv", index=False
    )
    print(f"\nSaved breakdown to {config.RESULTS_FIGURES_DIR / 'cwhs_breakdown_by_severity.csv'}")


Severity-Stratified Breakdown:
            Architecture  High Hal. Rate  High N  Medium Hal. Rate  Medium N  Low Hal. Rate  Low N   CWHS  High Corr.  Medium Corr.  Low Corr.    CSS
     Evidence-Graded RAG          0.0069      56            0.0237       122         0.0038     22 0.0162      0.7893        0.8262     0.9318 0.0828
     Query Decomposition          0.0182      56            0.0164       122         0.0000     22 0.0163      0.7607        0.7385     0.7636 0.1104
 Query Expansion (Multi)          0.0176      56            0.0114       122         0.0000     22 0.0132      0.7143        0.7230     0.8864 0.1168
  Hybrid + Cross-Encoder          0.0221      56            0.0194       122         0.0000     22 0.0195      0.7054        0.7311     0.8364 0.1211
           Naive RAG k=5          0.0186      56            0.0208       122         0.0227     22 0.0200      0.7071        0.7016     0.8091 0.1283
              Hybrid RRF          0.0210      56            0.0208  

## Per-Question CWHS and CSS Scores

Saves a per-question CSV for each architecture with individual CWHS and CSS contributions.
This enables question-level analysis — identifying which specific questions drive an
architecture's aggregate score, and whether clinically dangerous failures cluster on
particular topics or question types.

Columns:
- `question_idx`: index from the golden dataset (for cross-referencing with other results)
- `question`: the question text
- `severity_tier` / `severity_weight`: clinical risk classification
- `faithfulness`: FaithfulnessMetric score (0–1)
- `hallucination_rate`: 1 − faithfulness
- `cwhs_contribution`: (1 − f_i) × w_i — the weighted hallucination contribution
- `correctness`: AnswerCorrectness score (if available)
- `css_contribution`: α(1 − f_i)w_i + β(1 − c_i)w_i — the weighted combined risk (if available)

In [25]:
if per_question_dfs:
    pq_output_dir = config.RESULTS_FIGURES_DIR
    for arch_label, pq_df in per_question_dfs.items():
        prefix = [info["prefix"] for lbl, info in ARCHITECTURES.items() if lbl == arch_label][0]
        filename = f"cwhs_per_question_{prefix}.csv"
        pq_df.to_csv(pq_output_dir / filename, index=False)

    print(f"Saved {len(per_question_dfs)} per-question CSVs to {pq_output_dir}/")
    print()

    # Show a sample: top 10 highest-risk questions across all architectures
    print("Top 10 highest CSS-contribution questions (across all architectures):")
    print("-" * 90)
    for arch_label, pq_df in sorted(per_question_dfs.items()):
        if "css_contribution" in pq_df.columns:
            worst = pq_df.nlargest(3, "css_contribution")[["question_idx", "severity_tier", "faithfulness", "correctness", "css_contribution"]]
            print(f"\n  {arch_label}:")
            for _, row in worst.iterrows():
                print(f"    q{int(row['question_idx']):>3d} [{row['severity_tier']:>6s}]  "
                      f"faith={row['faithfulness']:.2f}  corr={row['correctness']:.2f}  "
                      f"css_contrib={row['css_contribution']:.4f}")

Saved 11 per-question CSVs to /content/results/figures/

Top 10 highest CSS-contribution questions (across all architectures):
------------------------------------------------------------------------------------------

  Evidence-Graded RAG:
    q 78 [  High]  faith=0.92  corr=0.10  css_contrib=1.2300
    q 59 [  High]  faith=1.00  corr=0.00  css_contrib=1.2000
    q 76 [  High]  faith=1.00  corr=0.00  css_contrib=1.2000

  Hybrid + Cross-Encoder:
    q167 [  High]  faith=0.89  corr=0.00  css_contrib=1.4000
    q 46 [  High]  faith=1.00  corr=0.00  css_contrib=1.2000
    q163 [  High]  faith=0.88  corr=0.20  css_contrib=1.1850

  Hybrid RRF:
    q168 [  High]  faith=0.86  corr=0.10  css_contrib=1.3371
    q 46 [  High]  faith=1.00  corr=0.00  css_contrib=1.2000
    q 59 [  High]  faith=1.00  corr=0.00  css_contrib=1.2000

  MeSH-Guided RAG:
    q129 [  High]  faith=0.50  corr=0.00  css_contrib=2.1000
    q163 [  High]  faith=0.86  corr=0.10  css_contrib=1.3371
    q167 [  High]  faith=

## Δ and CSS Interpretation

- **Δ (CWHS − Unweighted)**: positive = hallucinations concentrate on high-risk questions;
  negative = hallucinations concentrate on low-risk questions; zero = evenly distributed.
- **CSS ranking vs CWHS ranking**: differences reveal architectures whose answer quality
  is better or worse than their hallucination rate alone suggests.

In [26]:
if cwhs_results:
    print("Delta interpretation per architecture:")
    print("-" * 60)
    for arch, r in sorted(cwhs_results.items(), key=lambda x: x[1]["delta"]):
        delta = r["delta"]
        if delta > 0.002:
            label = "⚠ Worse on high-risk questions"
        elif delta < -0.002:
            label = "✓ Better on high-risk questions"
        else:
            label = "~ Evenly distributed"
        print(f"  {arch:35s}  Δ={delta:+.4f}  {label}")

    if css_results:
        print("\n")
        print("CSS vs CWHS ranking comparison:")
        print("-" * 75)
        cwhs_rank = {arch: rank + 1 for rank, (arch, _) in
                     enumerate(sorted(cwhs_results.items(), key=lambda x: x[1]["cwhs"]))}
        css_rank = {arch: rank + 1 for rank, (arch, _) in
                    enumerate(sorted(css_results.items(), key=lambda x: x[1]["css"]))}
        print(f"  {'Architecture':35s}  {'CWHS Rank':>10s}  {'CSS Rank':>9s}  {'Shift':>6s}")
        for arch in sorted(css_results, key=lambda a: css_results[a]["css"]):
            cr = cwhs_rank.get(arch, "-")
            cc = css_rank.get(arch, "-")
            shift = cr - cc if isinstance(cr, int) and isinstance(cc, int) else 0
            arrow = f"+{shift}" if shift > 0 else str(shift) if shift < 0 else "="
            print(f"  {arch:35s}  {cr:>10}  {cc:>9}  {arrow:>6s}")

Delta interpretation per architecture:
------------------------------------------------------------
  Naive RAG k=8                        Δ=-0.0011  ~ Evenly distributed
  Evidence-Graded RAG                  Δ=-0.0006  ~ Evenly distributed
  Naive RAG k=5                        Δ=-0.0003  ~ Evenly distributed
  Query Expansion (Single)             Δ=-0.0003  ~ Evenly distributed
  Naive RAG With Critic                Δ=+0.0005  ~ Evenly distributed
  Naive RAG k=3                        Δ=+0.0007  ~ Evenly distributed
  Query Decomposition                  Δ=+0.0012  ~ Evenly distributed
  Hybrid RRF                           Δ=+0.0013  ~ Evenly distributed
  Query Expansion (Multi)              Δ=+0.0013  ~ Evenly distributed
  Hybrid + Cross-Encoder               Δ=+0.0014  ~ Evenly distributed
  MeSH-Guided RAG                      Δ=+0.0021  ⚠ Worse on high-risk questions


CSS vs CWHS ranking comparison:
---------------------------------------------------------------------------